# BBarolo 2: the examples

This notebook runs the examples of this folder with the Python interface of BBarolo. Everything is computed in
memory: nothing is written on disk, and the results are shown as tables and plots. Each example has the setup of
its parameter file (`*.par`, which runs the same fit with `BBarolo -p <file>`), and is explained on its page of the
[documentation](https://bbarolo.readthedocs.io/en/latest/examples/index.html).

Run the notebook from this folder, where the FITS files are. The times are those of a server with 16 cores. The
examples marked *slow* take more than 10 minutes and run only with `RUN_SLOW = True` (below).

| Example | Data | Fit | Time (16 cores) |
|---|---|---|---|
| NGC 2403 | HI, VLA, nearby spiral | 3DFIT CLASSIC, geometry free, errors | 52 s |
| NGC 5633 | Hα + [NII], CALIFA optical IFU | CONTSUB, 3DFIT CLASSIC with three lines | 25 s |
| PGC 53639 | HI, ATCA, warped disk at low inclination | 3DFIT CLASSIC, position angle free | 24 s |
| UGC 1281 | HI, WSRT, edge-on dwarf | 3DFIT GLOBAL, smoothness priors, errors | 38 s |
| NGC 891 (*slow*) | HI, WSRT, edge-on spiral with a lagging halo | 3DFIT GLOBAL, thin disk and thick layer with a lag | 19 min |
| U3_25160 | Hα, KMOS, star-forming galaxy at z = 0.9 | 3DFIT BAYESIAN | 1 min 42 s |
| ALESS 073.1 | [CII], ALMA, starburst galaxy at z = 4.8 | 3DFIT BAYESIAN, prior on the inclination | 8 min |
| HD 142527 (*slow*) | ¹³CO, ALMA, protoplanetary disk | 3DFIT BAYESIAN, geometry and thickness free | 52 min |
| 2MASX J1022 | Hα + [NII], SALT long slit, massive spiral | SLITFIT GLOBAL | 23 s |
| Circinus (*slow*) | [OIII], MUSE, ionised nuclear wind of a Seyfert galaxy | WINDFIT GLOBAL with a mask | 16 min |

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np

try:
    import bbarolo
except ImportError:                     # not installed: the package of this repository
    sys.path.insert(0, os.path.abspath(".."))
    import bbarolo
from bbarolo import FitMod3D, SlitFit, WindFit

THREADS = min(16, os.cpu_count())       # threads (CLASSIC) or processes (GLOBAL, BAYESIAN) of the fits
RUN_SLOW = False                        # True: also NGC 891 (19 min), HD 142527 (52 min) and Circinus (16 min)
print(f"BBarolo {bbarolo.__version__}, {THREADS} threads")

## NGC 2403: HI with 3DFIT

The classic ring-by-ring fit of a nearby spiral observed with the VLA: rotation velocity, dispersion, position
angle and inclination of 42 rings, with errors. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/ngc2403.html).

In [ ]:
f = FitMod3D("ngc2403.fits", nradii=42, radsep=30, vsys=132.8, xpos=77, ypos=77, vrot=100, vrad=0,
             vdisp=8, inc=60, pa=123.7, z0=10, free="VROT VDISP PA INC", flagerrors=True)
res_n2403 = f.run(threads=THREADS)
res_n2403.rings                         # the rings of the second stage, with the errors

In [ ]:
res_n2403.plot_parameters()      # the rings against radius
res_n2403.plot_kinmaps()         # moment maps: data, model, residuals
res_n2403.plot_pvs()             # position-velocity diagrams along the major and minor axes
res_n2403.plot_pvslices()        # position-velocity slices at several angles and offsets
res_n2403.plot_chanmaps()        # channel maps: data and model
plt.show()

## NGC 5633: Hα and [NII] in an optical IFU cube

A CALIFA cube: the continuum is subtracted first (`contsub=True`), then Hα and the two [NII] lines are fitted
together. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/ngc5633.html).

In [ ]:
f = FitMod3D("NGC5633_V500.fits", contsub=True, exclwind="37:68", contorder=1,
             beam=1.7, specres=1.2, veldef="OPTICAL",
             restwave=[6564.61, 6549.86, 6585.27], relint=[1, 0.15, 0.46], redshift=0.00757,
             nradii=17, radsep=1.7, xpos=35.0, ypos=32.6, inc=50, pa=199, vrot=150, vdisp=25, vsys=-33, z0=1,
             mask="SEARCH", threshold=0.2, growththreshold=0.03, threshvelocity=5,
             free="VROT VDISP", norm="AZIM", minvdisp=15, flagerrors=True, maptype="GAUSSIAN")
res_n5633 = f.run(threads=THREADS)
res_n5633.rings

In [ ]:
res_n5633.plot_parameters()      # the rings against radius
res_n5633.plot_kinmaps()         # moment maps: data, model, residuals
res_n5633.plot_pvs()             # position-velocity diagrams along the major and minor axes
res_n5633.plot_pvslices()        # position-velocity slices at several angles and offsets
res_n5633.plot_chanmaps()        # channel maps: data and model
plt.show()

## PGC 53639: a warped disk seen nearly face-on

An HI cube of the LVHIS survey (ATCA): the inclination is fixed, the position angle is free in each ring and
traces the warp. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/pgc53639.html).

In [ ]:
f = FitMod3D("pgc53639.fits", specres=0.425, nradii=16, radsep=30, inc=25, pa=150, vsys=589, xpos=61.9,
             ypos=63.1, free="VROT VDISP PA", mask="SEARCH", minvdisp=5, deltapa=50, flagerrors=True)
res_p53639 = f.run(threads=THREADS)
res_p53639.rings

In [ ]:
res_p53639.plot_parameters()      # the rings against radius
res_p53639.plot_kinmaps()         # moment maps: data, model, residuals
res_p53639.plot_pvs()             # position-velocity diagrams along the major and minor axes
res_p53639.plot_pvslices()        # position-velocity slices at several angles and offsets
res_p53639.plot_chanmaps()        # channel maps: data and model
plt.show()

## UGC 1281: a global fit of an edge-on dwarf

In an edge-on disk the rings overlap along the line of sight, so they are fitted together: `method="global"`
maximises the likelihood of all the parameters at once, with smoothness priors on the profiles and errors from the
Laplace approximation. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/ugc1281.html).

In [ ]:
f = FitMod3D("ugc1281.fits", method="global", specres=0.425, nradii=8, radsep=30, inc=85,
             free="vrot vdisp inc_single phi_single z0_single xpos_single ypos_single vsys_single radmax fracerr",
             mask="SEARCH", flagerrors=True, likeopts="smooth=(vrot=2.3,vdisp=1)")
res_u1281 = f.run(threads=THREADS)
res_u1281.fitter.print_stats()          # the free parameters, with the errors

In [ ]:
res_u1281.plot_parameters()      # the rings against radius
res_u1281.plot_kinmaps()         # moment maps: data, model, residuals
res_u1281.plot_pvs()             # position-velocity diagrams along the major and minor axes
res_u1281.plot_pvslices()        # position-velocity slices at several angles and offsets
res_u1281.plot_chanmaps()        # channel maps: data and model
res_u1281.fitter.corner_plot()   # the draws of the Laplace approximation
plt.show()

## NGC 891: the lagging HI halo of an edge-on spiral (*slow*)

A global fit with two layers: a thin disk and a thick layer (the halo) whose rotation decreases with the height
above the plane. 19 minutes on 16 cores: it runs only with `RUN_SLOW = True`.
[Documentation](https://bbarolo.readthedocs.io/en/latest/examples/ngc891.html).

In [ ]:
f = FitMod3D("ngc891.fits", method="global", specres=0.825, nradii=16, radsep=40,
             xpos="2:22:32.99", ypos="+42:20:52.1", inc=88, pa=203, vsys=528, vrot=180, vdisp=13, z0=3,
             radmaxa=440, z0thick=30, fthick=0.3, vdispthick=32, dvdz=0.8, mask="SEARCH", ltype=3,
             free="vrot vdisp_single z0_single z0thick_single fthick_single vdispthick_single dvdz_single "
                  "fracerr radmaxa",
             priors="vrot=(uniform,30,200) dvdz=(uniform,loc=0,scale=2) radmaxa=(uniform,loc=400,scale=100)",
             flagerrors=True)
res_n891 = f.run(threads=THREADS) if RUN_SLOW else None
if res_n891 is not None:
    res_n891.fitter.print_stats()

In [ ]:
if res_n891 is not None:
    res_n891.plot_parameters()      # the rings against radius
    res_n891.plot_kinmaps()         # moment maps: data, model, residuals
    res_n891.plot_pvs()             # position-velocity diagrams along the major and minor axes
    res_n891.plot_pvslices()        # position-velocity slices at several angles and offsets
    res_n891.plot_chanmaps()        # channel maps: data and model
    res_n891.fitter.corner_plot()
    plt.show()

## U3_25160: a Bayesian fit of Hα at z = 0.9

A KMOS cube of a star-forming galaxy (KMOS<sup>3D</sup>): `method="bayesian"` samples the posterior of the
rotation curve, the dispersion and the geometry with nested sampling. The sampler is random: the numbers change
slightly from run to run. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/u3-25160.html).

In [ ]:
f = FitMod3D("U3_25160_Ha.fits", method="bayesian", beam=0.65, specres=0.90, redshift=0.89615,
             restwave=0.656461, nradii=4, radsep=0.3, vsys=4, xpos=11.6, ypos=12.3, vrot=130, vdisp=20,
             inc=50, pa=305, z0=0.1,
             free="VROT VDISP INC_SINGLE PA_SINGLE XPOS_SINGLE YPOS_SINGLE VSYS_SINGLE FRACERR",
             priors="vdisp=(uniform,loc=0,scale=100) vrot=(uniform,loc=0,scale=250)",
             mask="SEARCH", growthcut=1.8)
res_u3 = f.run(threads=THREADS)
res_u3.fitter.print_stats()             # medians and 68% intervals

In [ ]:
res_u3.plot_parameters()      # the rings against radius
res_u3.plot_kinmaps()         # moment maps: data, model, residuals
res_u3.plot_pvs()             # position-velocity diagrams along the major and minor axes
res_u3.plot_pvslices()        # position-velocity slices at several angles and offsets
res_u3.plot_chanmaps()        # channel maps: data and model
res_u3.fitter.corner_plot()      # the posterior
plt.show()

## ALESS 073.1: a Bayesian fit of [CII] at z = 4.76

An ALMA cube of a starburst galaxy 1.2 billion years after the Big Bang, nearly face-on: a Gaussian prior on the
inclination, from Lelli et al. (2021), breaks its degeneracy with the rotation velocity. About 8 minutes on 16
cores. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/aless73.html).

In [ ]:
f = FitMod3D("ALESS73_CII.fits", method="bayesian", specres=0.715, radii="0.03~0.5:0.06",
             free="vrot vdisp xpos_single ypos_single vsys_single inc_single phi_single fracerr",
             priors="vrot=(uniform,0,600) vdisp=(norm,50,30) inc=(norm,25,2)")
res_a73 = f.run(threads=THREADS)
res_a73.fitter.print_stats()

In [ ]:
res_a73.plot_parameters()      # the rings against radius
res_a73.plot_kinmaps()         # moment maps: data, model, residuals
res_a73.plot_pvs()             # position-velocity diagrams along the major and minor axes
res_a73.plot_pvslices()        # position-velocity slices at several angles and offsets
res_a73.plot_chanmaps()        # channel maps: data and model
res_a73.fitter.corner_plot()
plt.show()

## HD 142527: a Bayesian fit of a protoplanetary disk (*slow*)

The ¹³CO disk around a young star (ALMA), with the geometry and the thickness free. This example also shows how
to set or change the parameters after the constructor, with `update()`. 52 minutes on 16 cores: it runs only
with `RUN_SLOW = True`. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/hd142527.html).

In [ ]:
f = FitMod3D("HD142527_13CO.fits", method="bayesian", radii=[0.65, 1.10, 1.55, 2.00, 2.45, 2.90])

# Parameters can be set or changed after the constructor: update() returns the task
f.update(specres=0.425,
         free="VROT VDISP PA_SINGLE INC_SINGLE XPOS_SINGLE YPOS_SINGLE VSYS_SINGLE Z0_SINGLE RADMAX FRACERR")
f.update(priors={"vrot": {"name": "uniform", "loc": 0, "scale": 20},    # or ("uniform", 0, 20), or the string
                 "vdisp": {"name": "uniform", "loc": 0, "scale": 10},   # of the parameter file
                 "inc": {"name": "uniform", "loc": 10, "scale": 20},
                 "vsys": {"name": "uniform", "loc": -10, "scale": 20}})
print(f.parameters)                     # all the parameters set so far

res_hd = f.run(threads=THREADS) if RUN_SLOW else None
if res_hd is not None:
    res_hd.fitter.print_stats()

In [ ]:
if res_hd is not None:
    res_hd.plot_parameters()      # the rings against radius
    res_hd.plot_kinmaps()         # moment maps: data, model, residuals
    res_hd.plot_pvs()             # position-velocity diagrams along the major and minor axes
    res_hd.plot_pvslices()        # position-velocity slices at several angles and offsets
    res_hd.plot_chanmaps()        # channel maps: data and model
    res_hd.fitter.corner_plot()
    plt.show()

In [ ]:
if res_hd is not None:
    # the mass of the central stars from the Keplerian rotation, at 159.3 pc (see the documentation)
    G = 887.13                                  # the gravitational constant in (km/s)^2 au / Msun
    R = np.array([0.65, 1.10, 1.55, 2.00, 2.45, 2.90]) * 159.3   # the radii in au, at 159.3 pc
    s, names = res_hd.fitter.samples_equal, res_hd.fitter.freepar_names   # the posterior samples (equal weights)
    vrot = s[:, [names.index(f"vrot{i + 1}") for i in range(len(R))]]
    inc = np.radians(s[:, names.index("inc")])
    M = (vrot ** 2 * R).mean(axis=1) / G        # the mass of each sample (Msun)
    print(np.percentile(M, [16, 50, 84]))                       # M
    print(np.percentile(M * np.sin(inc) ** 2, [16, 50, 84]))    # M sin^2 i

## 2MASX J1022: a long-slit spectrum with SLITFIT

A SALT long-slit spectrum of a massive spiral at z = 0.09: SLITFIT models the light of the disk that falls into
the slit, after subtracting the continuum of the stars. [Documentation](https://bbarolo.readthedocs.io/en/latest/examples/2masxj1022.html).

In [ ]:
f = SlitFit("2masxj1022_SALT.fits", method="global", slitwidth=1, beam=1.5, specres=1.7, redshift=0.09108,
            restwave=[6564.61, 6549.86, 6585.27], relint=[1, 0.17, 0.45], contsub=True, skyresid=True,
            radii="range(2.5,19,2)", inc=71, xpos=110, norm="AZIM", free="VROT VDISP VSYS",
            priors="vrot=(uniform,loc=100,scale=350) vdisp=(uniform,loc=10,scale=150) "
                   "vsys=(uniform,loc=-10,scale=20)",
            likeopts="smooth=(vrot=50,vdisp=1)")
res_j1022 = f.run(threads=THREADS)
res_j1022.rings

In [ ]:
res_j1022.fitter.plot_parameters()      # the rings against radius
res_j1022.fitter.plot_pv()              # the slit, the best model and the residuals
plt.show()

## Circinus: an ionised nuclear wind (*slow*)

The [OIII] cone of the Circinus galaxy (MUSE) fitted with WINDFIT, with the geometry of Marconcini et al. (2023) and a
mask of the cone (`circinus_mask.py` makes `circinus_OIII_mask.fits`). 16 minutes on 16 cores: it runs only with
`RUN_SLOW = True`. The maps and position-velocity plots of WINDFIT are written by `res_circ.write()`.
[Documentation](https://bbarolo.readthedocs.io/en/latest/examples/circinus.html).

In [ ]:
f = WindFit("circinus_OIII.fits", method="global", mask="FILE(circinus_OIII_mask.fits)", restwave=5006.843,
            redshift=0.001448, specres=0.85, xpos="14:13:09.90", ypos="-65:20:19.77", vsys=0, inc=70, pa=25,
            openang=110, conelen=52, vwind=400, vdisp=0, free="VWIND_SINGLE INC",
            priors="INC=(uniform,30,60) VWIND=(uniform,0,1600)", sampleropts="nstart=6 seed=1", flagerrors=True,
            likeopts="emission=cells cellsize=3.6 datafrac=0.3 cones=approaching likemask=dilated dilation=0,0")
res_circ = f.run(threads=THREADS) if RUN_SLOW else None
if res_circ is not None:
    res_circ.fitter.print_stats()

In [ ]:
if res_circ is not None:
    print(res_circ.shells)          # the shells of the best fit
    res_circ.fitter.corner_plot()   # the Laplace approximation around the best fit
    plt.show()